<a href="https://colab.research.google.com/github/seunghyeon-hwang/cat-dog-image-classifier/blob/main/cat-dog-image-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%rm -rf dataset
%rm -rf train
%rm -rf dogs-vs-cats-redux-kernels-edition.zip
%rm -rf test.zip
%rm -rf sample_submission.csv
%rm -rf test.zip
%rm -rf train.zip
%rm -rf log

In [3]:
### 개고양이 구분 AI 만들기
import os
import tensorflow as tf
import shutil
from google.colab import drive
import time
import datetime
from tensorflow.keras.callbacks import TensorBoard, Callback, EarlyStopping

drive.mount('/content/drive')

kaggle_path = '/content/drive/MyDrive/Colab Notebooks/data/kaggle.json'
os.environ['KAGGLE_CONFIG_DIR'] = os.path.dirname(kaggle_path)


!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition

!unzip -q dogs-vs-cats-redux-kernels-edition.zip -d .

!unzip -q train.zip -d .

os.mkdir('/content/dataset')
os.mkdir('/content/dataset/cat')
os.mkdir('/content/dataset/dog')

for i in os.listdir('/content/train/'):
  if 'cat' in i:
    shutil.copyfile('/content/train/'+i, '/content/dataset/cat/'+i)
  elif 'dog' in i:
    shutil.copyfile('/content/train/'+i, '/content/dataset/dog/'+i)

Mounted at /content/drive
100% 814M/814M [00:19<00:00, 42.7MB/s]



In [4]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/dataset/', # 이미지가 저장된 디렉토리 경로를 지정합니다.
    image_size=(64, 64), # 이미지 크기를 64x64 픽셀로 재조정합니다.
    batch_size=64, # 한 번에 처리할 이미지 수를 64로 설정합니다.
    subset='training', # 데이터셋을 훈련용으로 사용합니다.
    validation_split=0.2, # 전체 데이터의 20%를 검증 데이터로 분할합니다.
    seed=1234 # 데이터 분할의 재현성을 위한 시드 값을 설정합니다.
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/dataset/', # 이미지가 저장된 디렉토리 경로를 지정합니다.
    image_size=(64, 64), # 이미지 크기를 64x64 픽셀로 재조정합니다.
    batch_size=64, # 한 번에 처리할 이미지 수를 64로 설정합니다.
    subset='validation', # 데이터셋을 검증용으로 사용합니다.
    validation_split=0.2, # 전체 데이터의 20%를 검증 데이터로 분할합니다.
    seed=1234 # 데이터 분할의 재현성을 위한 시드 값을 설정합니다.
)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [5]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds_scaled = train_ds.map(preprocess)
val_ds_scaled = val_ds.map(preprocess)

Image pixel values scaled to [0, 1].


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', input_shape=(64,64,3)), # 이미지를 수평으로 무작위 뒤집기 (데이터 증강 기법)
    tf.keras.layers.RandomRotation(0.1), # 이미지를 10% 범위 내에서 무작위로 회전 (데이터 증강 기법)
    tf.keras.layers.RandomZoom(0.1), # 이미지를 10% 범위 내에서 무작위로 확대/축소 (데이터 증강 기법)
    tf.keras.layers.Conv2D(32, (3,3), padding='same', activation='relu'), # 32개의 3x3 필터(커널)를 사용하는 합성곱 레이어. 'same' 패딩으로 출력 이미지 크기 유지, 'relu' 활성화 함수 사용
    tf.keras.layers.MaxPooling2D((2,2)), # 2x2 크기의 맥스 풀링 레이어. 특징 맵의 크기를 절반으로 줄여 계산량을 감소시키고 주요 특징을 강조
    tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.2), # 20%의 확률로 무작위 뉴런을 비활성화하여 과적합을 방지하는 Dropout 레이어
    tf.keras.layers.Flatten(), # 이전 레이어의 다차원 특징 맵을 1차원 벡터로 평탄화하여 완전 연결 레이어(Dense layer)에 입력할 수 있도록 준비
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])


model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 텐서보드 콜백 설정: 모델 훈련 과정을 시각화하기 위해 로그를 저장할 디렉토리 지정
tensorboard = TensorBoard(log_dir='/content/logs/{}'.format('모델' + str(int(time.time()))))

# EarlyStopping 콜백 설정: 검증 손실(val_loss)을 모니터링하여, 10 에포크 동안 검증 손실이 개선되지 않으면 훈련을 자동으로 중단 ('min' 모드는 손실 감소를 의미)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

model.fit(train_ds_scaled, validation_data=val_ds_scaled, epochs=5, callbacks=[tensorboard, es])

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 206s 648ms/step - accuracy: 0.5573 - loss: 0.6865 - val_accuracy: 0.6040 - val_loss: 0.6514
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 260s 645ms/step - accuracy: 0.6722 - loss: 0.6059 - val_accuracy: 0.6640 - val_loss: 0.6519
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 195s 621ms/step - accuracy: 0.7127 - loss: 0.5573 - val_accuracy: 0.6928 - val_loss: 0.5921
Epoch 4/5
308/313 ━━━━━━━━━━━━━━━━━━━━ 2s 551ms/step - accuracy: 0.7265 - loss: 0.5382

In [6]:
%load_ext tensorboard
%tensorboard --logdir logs

ModuleNotFoundError: No module named 'tensorboard # 텐서보드 Jupyter 확장 기능을 로드합니다'